# 模型评估与对比
## AI电商评论分析系统 — BERT vs LLM 双路线对比
本 notebook 用于评估训练好的模型并对比 BERT 和蒸馏 LLM 的性能差异。

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

from nlp.src.evaluation.evaluate import ModelEvaluator
from nlp.src.evaluation.confusion import confusion_to_dict, find_confused_pairs
from nlp.src.evaluation.metrics import classification_metrics

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
%matplotlib inline

In [ ]:
# BERT 评估结果（示例）
bert_predictions = []  # 替换为实际预测
true_labels = []       # 替换为真实标签

evaluator = ModelEvaluator()
bert_results = evaluator.evaluate(true_labels, bert_predictions)
print('BERT Results:')
print(json.dumps(bert_results, indent=2, ensure_ascii=False))

In [ ]:
# 混淆矩阵可视化
cm = np.array(bert_results.get('confusion_matrix', [[0,0,0],[0,0,0],[0,0,0]]))
labels = bert_results.get('per_class', {}).keys() if 'per_class' in bert_results else ['negative', 'neutral', 'positive']

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# LLM 结果（示例）
llm_results = evaluator.evaluate(true_labels, llm_predictions) if 'llm_predictions' in dir() else None
if llm_results:
    comparison = evaluator.compare_models(bert_results, llm_results)
    print('\nModel Comparison:')
    print(json.dumps(comparison, indent=2, ensure_ascii=False))

In [ ]:
# Bad Case 分析
from nlp.src.evaluation.error_analysis import ErrorAnalyzer

analyzer = ErrorAnalyzer()
bad_cases = analyzer.analyze_bad_cases(texts, true_labels, bert_predictions) if 'texts' in dir() else []
summary = analyzer.summarize_errors(bad_cases)
print('Error Summary:', json.dumps(summary, indent=2, ensure_ascii=False))

In [ ]:
# 总结
print('=== 评估总结 ===')
print(f'BERT Accuracy: {bert_results.get("accuracy", 0):.2%}')
print(f'BERT F1: {bert_results.get("f1_weighted", 0):.2%}')
if llm_results:
    print(f'LLM Accuracy: {llm_results.get("accuracy", 0):.2%}')
    print(f'LLM F1: {llm_results.get("f1_weighted", 0):.2%}')
    print(f'Delta F1: {comparison["delta"]["f1"]:+.2%}')